# Naive fixed-rule baseline

This notebook implements a naive baseline to aggregate experimental readouts of COVID-19's screening assays. For each screen it reads the *same raw input files as the RepurAgent received*, take mean aggregation naively.

The results are stored to `naive_scoring/`, so that `COVID-19_analysis.ipynb` can draw the AUC-ROC figures.

In [2]:
import os
import numpy as np
import pandas as pd

# create outputfolder 
os.makedirs('naive_scoring', exist_ok=True)

# Naive mean aggregation function
def naive_composite_score(df, specs):
    """
    Each assay column is standardized to scale 0-1 (min-max scaling),
    oriented so that higher = more hit-like, then averaged across the assays available for each compound. 

    Parameters
    ----------
    df : pandas DataFrame
    specs : list of (column_name, higher_is_better) tuples
        higher_is_better=True  -> larger value = more hit-like (kept as +z)
        higher_is_better=False -> smaller value = more hit-like (flipped to -z)

    Returns
    -------
    pd.Series of composite scores (higher = better). Missing assay values are
    ignored in the per-compound average (nan-mean).
    """
    z_cols = []
    for col, higher_is_better in specs:
        x = df[col].astype(float)
        z = (x - x.min()) / (x.max() - x.min())
        if not higher_is_better:
            z = -z
        z_cols.append(z.values)

    composite = np.nanmean(np.vstack(z_cols), axis=0)
    return pd.Series(composite, index=df.index)

## Primary screen

Raw inputs:
* `CP_AB_raw.xlsx`: `Morphology score` (higher is better) and `Infection rate (%)` (lower is better), one row per compound/concentration.
* `CPE_raw.xlsx`: `Inhibition of cytopathicity (%)` (higher is better) measured across a dose range (about 5 concentrations per compound).

Naive aggregation averages every readout over all rows per compound (for CPE that means averaging the whole dose range).

In [3]:
cpab = pd.read_excel('raw_data/CP_AB_raw.xlsx')[['Compound_name', 'Morphology score', 'Infection rate (%)']]
cpab = cpab.rename(columns={'Compound_name': 'compound_name'})
cpe  = pd.read_excel('raw_data/CPE_raw.xlsx')[['Compound_name', 'Inhibition of cytopathicity (%)']]
cpe  = cpe.rename(columns={'Compound_name': 'compound_name'})

# Naive aggregation: mean per compound (CPE mean = averaged over the dose range)
cpab = cpab.groupby('compound_name', as_index=False)[['Morphology score', 'Infection rate (%)']].mean()
cpe  = cpe.groupby('compound_name', as_index=False)['Inhibition of cytopathicity (%)'].mean()
naive = cpab.merge(cpe, on='compound_name', how='outer')

# Naive scoring: equal-weight z-score of the three assays
naive['naive_composite'] = naive_composite_score(
    naive,
    [('Morphology score', True),
     ('Inhibition of cytopathicity (%)', True),
     ('Infection rate (%)', False)],
)
naive['naive_rank'] = naive['naive_composite'].rank(ascending=False, method='min')
naive = naive.sort_values('naive_rank').reset_index(drop=True)

# Store naive results
naive.to_csv('naive_scoring/primary_screen_naive.csv', index=False)
naive.head()

,compound_name,Morphology score,Infection rate (%),Inhibition of cytopathicity (%),naive_composite,naive_rank
0,GC376 sodium,NaN,NaN,13.932,0.666751,1.0
1,PF-02545920,0.927244,8.0,14.008,0.543717,2.0
2,calpeptin,0.814525,12.5,15.106,0.513482,3.0
3,elesclomol,0.879490,19.5,15.624,0.512797,4.0
4,spiperone,0.762606,22.0,19.040,0.503395,5.0


## Validation screen

Raw inputs: 
- `validatation_experiements.xlsx` (`morphology_score` higher is better; `Infection rate (%)` lower is better)
- `phospholipidosis_raw.xlsx` (`24h_DIPL(%)` lower is safer)

Each readout is averaged over all tested concentrations per compound.

In [4]:
val = pd.read_excel('raw_data/validatation_experiements.xlsx')[['Compound_name', 'morphology_score', 'Infection rate (%)']]
val = val.rename(columns={'Compound_name': 'compound_name'})
pld = pd.read_excel('raw_data/phospholipidosis_raw.xlsx')[['compound_name', '24h_DIPL(%)']]

# Naive aggregation: average each readout over all tested concentrations per compound
val = val.groupby('compound_name', as_index=False).mean(numeric_only=True)
pld = pld.groupby('compound_name', as_index=False)['24h_DIPL(%)'].mean()

naive = val.merge(pld, on='compound_name', how='outer')

# Naive scoring: equal-weight z-score (morphology up, infection down, DIPL down)
naive['naive_composite'] = naive_composite_score(
    naive,
    [('morphology_score', True),
     ('Infection rate (%)', False),
     ('24h_DIPL(%)', False)],
)
naive['naive_rank'] = naive['naive_composite'].rank(ascending=False, method='min')
naive = naive.sort_values('naive_rank').reset_index(drop=True)

# Store naive result
naive.to_csv('naive_scoring/validation_screen_naive.csv', index=False)
naive.head()

,compound_name,morphology_score,Infection rate (%),24h_DIPL(%),naive_composite,naive_rank
0,bruceantin,0.2825,15.00,NaN,0.238471,1.0
1,anisomycin,0.6100,11.25,1.865000,0.201914,2.0
2,emetine,0.6100,13.00,4.735000,0.191550,3.0
3,Cathepsin Inhibitor 1,0.8475,31.50,13.416667,0.184819,4.0
4,bortezomib,0.2775,26.25,NaN,0.180765,5.0
